![image info](https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/images/banner_1.png)

# Proyecto 2 - Clasificación de género de películas (Pipeline Optimizado)

**Métrica de evaluación:** Macro ROC-AUC Score
**Baseline:** `CountVectorizer(max_features=1000)` + `OneVsRest(RandomForestClassifier)` → **0.7812**
**Objetivo:** Superar el baseline mediante preprocesamiento avanzado, TF-IDF y Regresión Logística regularizada.


In [ ]:
import warnings
warnings.filterwarnings('ignore')


In [ ]:
import re
import numpy as np
import pandas as pd

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split


## 1. Preprocesamiento Avanzado de Texto

In [ ]:
# Descargar recursos NLTK requeridos
for resource in ('stopwords', 'wordnet', 'omw-1.4', 'punkt'):
    nltk.download(resource, quiet=True)

STOP_WORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()


In [ ]:
def clean_text(text: str) -> str:
    """
    Pipeline de limpieza:
      1. Conversión a minúsculas
      2. Eliminación de etiquetas HTML
      3. Retención solo de caracteres alfabéticos
      4. Tokenización, eliminación de stop-words y lematización
    """
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', ' ', text)          # strip HTML
    text = re.sub(r'[^a-z\s]', ' ', text)         # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [
        lemmatizer.lemmatize(t)
        for t in tokens
        if t not in STOP_WORDS and len(t) > 2
    ]
    return ' '.join(tokens)


## 2. Carga de Datos

In [ ]:
dataTraining = pd.read_csv(
    'https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTraining.zip',
    encoding='UTF-8', index_col=0
)
dataTesting = pd.read_csv(
    'https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTesting.zip',
    encoding='UTF-8', index_col=0
)

print(f"Muestras entrenamiento : {len(dataTraining)}")
print(f"Muestras prueba        : {len(dataTesting)}")
dataTraining.head()


In [ ]:
# Aplicar pipeline de limpieza a plots de entrenamiento y prueba
dataTraining['plot_clean'] = dataTraining['plot'].apply(clean_text)
dataTesting['plot_clean']  = dataTesting['plot'].apply(clean_text)

# Comparación de un plot original vs. limpio
print("ORIGINAL:\n", dataTraining['plot'].iloc[0])
print("\nLIMPIO:\n", dataTraining['plot_clean'].iloc[0])


## 3. Codificación de la Variable Objetivo

In [ ]:
dataTraining['genres'] = dataTraining['genres'].map(lambda x: eval(x))
mlb = MultiLabelBinarizer()
y_genres = mlb.fit_transform(dataTraining['genres'])

print(f"Clases ({len(mlb.classes_)}): {list(mlb.classes_)}")
print(f"Dimensión matriz objetivo: {y_genres.shape}")


## 4. Ingeniería de Características — TF-IDF con N-Gramas

**Mejoras sobre CountVectorizer básico:**
- `max_features=15 000` → vocabulario 15× más rico
- `ngram_range=(1, 2)` → captura frases de 2 palabras (e.g., *serial killer*, *science fiction*)
- `sublinear_tf=True` → aplica log(1+tf), reduciendo el peso de términos muy frecuentes
- `min_df=2` → ignora términos que aparecen en un solo documento (ruido)


In [ ]:
tfidf = TfidfVectorizer(
    max_features=15_000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    analyzer='word',
    token_pattern=r'\b[a-z][a-z]+\b'
)

# fit solo en entrenamiento
X_dtm = tfidf.fit_transform(dataTraining['plot_clean'])
print(f"Dimensión matriz TF-IDF (entrenamiento): {X_dtm.shape}")


## 5. Separación Entrenamiento / Validación

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_dtm, y_genres, test_size=0.20, random_state=42
)

print(f"Entrenamiento : {X_train.shape}")
print(f"Validación    : {X_val.shape}")


## 6. Modelo — OneVsRest + Regresión Logística (L2)

**¿Por qué Regresión Logística en lugar de Random Forest?**

| Aspecto | Random Forest | Logistic Regression (L2) |
|---|---|---|
| Manejo de matrices sparse | Ineficiente (muchos ceros irrelevantes) | Nativo, muy eficiente |
| Velocidad de entrenamiento | Lento (100 árboles × 24 clases) | Rápido (gradient descent) |
| Performance en NLP | Subóptimo en alta dimensionalidad | State-of-the-art en TF-IDF clásico |
| Hiperparámetro clave | `max_depth`, `n_estimators` | `C` (fuerza de regularización) |


In [ ]:
lr = LogisticRegression(
    C=4.0,                    # regularización L2 (inverso de lambda)
    max_iter=1000,
    solver='lbfgs',
    class_weight='balanced',  # compensa clases desbalanceadas (e.g., 'News')
    random_state=42,
    n_jobs=-1
)

clf = OneVsRestClassifier(lr, n_jobs=-1)
clf.fit(X_train, y_train)


## 7. Evaluación en Validación

In [ ]:
y_val_pred = clf.predict_proba(X_val)
val_roc = roc_auc_score(y_val, y_val_pred, average='macro')

print(f"Validación Macro ROC-AUC : {val_roc:.4f}")
print(f"Baseline                 : 0.7812")
print(f"Mejora absoluta          : +{val_roc - 0.7812:.4f}")


## 8. Re-entrenamiento con Dataset Completo

In [ ]:
# Re-entrenamos con TODOS los datos de entrenamiento para maximizar
# la información disponible antes de predecir el conjunto de Kaggle
clf_final = OneVsRestClassifier(
    LogisticRegression(C=4.0, max_iter=1000, solver='lbfgs',
                       class_weight='balanced', random_state=42, n_jobs=-1),
    n_jobs=-1
)
clf_final.fit(X_dtm, y_genres)
print("Re-entrenamiento completado sobre todos los datos de entrenamiento.")


## 9. Predicción sobre el Conjunto de Prueba (Kaggle)

In [ ]:
# Verificación: se usa .transform() con el vocabulario ajustado en entrenamiento
X_test_dtm = tfidf.transform(dataTesting['plot_clean'])
print(f"Dimensión matriz prueba     : {X_test_dtm.shape}")
print(f"Dimensión matriz entrenamiento: {X_dtm.shape}")

assert X_test_dtm.shape[1] == X_dtm.shape[1], "¡ERROR: dimensiones de features no coinciden!"
print("✓ Dimensiones coherentes — se usó .transform() correctamente")


## 10. Generación del Archivo de Entrega (Kaggle)

In [ ]:
cols = [
    'p_Action', 'p_Adventure', 'p_Animation', 'p_Biography', 'p_Comedy',
    'p_Crime', 'p_Documentary', 'p_Drama', 'p_Family', 'p_Fantasy',
    'p_Film-Noir', 'p_History', 'p_Horror', 'p_Music', 'p_Musical',
    'p_Mystery', 'p_News', 'p_Romance', 'p_Sci-Fi', 'p_Short', 'p_Sport',
    'p_Thriller', 'p_War', 'p_Western'
]

y_pred_test = clf_final.predict_proba(X_test_dtm)

# Verificación de alineación entre clases del MLB y columnas del CSV
expected_genres = [c.replace('p_', '') for c in cols]
assert list(mlb.classes_) == expected_genres, f"Desalineación de géneros: {list(mlb.classes_)}"
assert y_pred_test.shape == (len(dataTesting), len(cols))

res = pd.DataFrame(y_pred_test, index=dataTesting.index, columns=cols)
res.to_csv('pred_genres_text_LR_tfidf.csv', index_label='ID')

print("Archivo guardado: pred_genres_text_LR_tfidf.csv")
res.head()
